# 15 - Structural-Break Test of the Sentiment Long-Short Alpha

**Goal:** Test formally whether the risk-adjusted alpha of the sentiment long-short
spread changed around 2020, rather than reading the regime split by eye. Two tests:
(1) a Chow test at the a-priori break date (January 2020), for both an alpha-only break
and a full break in all six factor loadings; (2) a sup-Wald (QLR) test for a break at an
unknown date, which lets the data choose the break month.

**Inputs:** `data/long_short_returns.parquet`, `data/factors.parquet` (2013-2023).
**Output:** `output/table_structural_break.csv`.

**Read:** if the break is not statistically significant, the 48-month post-period simply
lacks the power to date it, which supports describing the regime split as context rather
than a tested break. This notebook is a sensitivity check and changes no headline result.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path
import statsmodels.api as sm

DATA = Path.home()/"thesis"/"data"; OUTPUT = Path.home()/"thesis"/"output"
FAC = ["mkt_rf","smb","hml","rmw","cma","mom"]
ls = pd.read_parquet(DATA/"long_short_returns.parquet"); ls["date"]=pd.to_datetime(ls["date"])
fac = pd.read_parquet(DATA/"factors.parquet")
df = ls.merge(fac[["date"]+FAC], on="date", how="inner")
df = df[(df["date"].dt.year>=2013)&(df["date"].dt.year<=2023)].sort_values("date").reset_index(drop=True)
y = df["long_short"].astype("float64")
print(f"Verify kernel: {__import__('sys').executable}")
print(f"Sample: {len(df)} months, {df['date'].min().date()} to {df['date'].max().date()}")

Verify kernel: /opt/anaconda3/envs/thesis/bin/python
Sample: 132 months, 2013-01-31 to 2023-12-31


In [2]:
# (1) CHOW TEST at the a-priori break date: January 2020
df["post"] = (df["date"].dt.year >= 2020).astype(float)

# (1a) Alpha-only break: add a post-2020 dummy to the six-factor model
Xa = sm.add_constant(df[FAC].astype("float64")); Xa["post"] = df["post"]
ra = sm.OLS(y, Xa).fit(cov_type="HAC", cov_kwds={"maxlags":3})
d_alpha = ra.params["post"]
print("=== (1a) Chow test at 2020 - ALPHA break ===")
print(f"  pre-2020 alpha  = {ra.params['const']*12*100:+.2f}%/yr")
print(f"  post-2020 alpha = {(ra.params['const']+d_alpha)*12*100:+.2f}%/yr")
print(f"  change in alpha = {d_alpha*12*100:+.2f}%/yr  (t={ra.tvalues['post']:+.2f}, p={ra.pvalues['post']:.3f})")
sig_a = "SIGNIFICANT" if ra.pvalues['post']<0.05 else "not significant"
print(f"  -> alpha break is {sig_a} at 5%")

# (1b) Full break: interact the constant and all six factors with the post dummy
Xf = pd.DataFrame({"const":1.0}, index=df.index)
for c in FAC: Xf[c]=df[c].astype("float64")
Xf["post"]=df["post"]
for c in FAC: Xf[c+"_x_post"]=df[c].astype("float64")*df["post"]
rf = sm.OLS(y, Xf).fit(cov_type="HAC", cov_kwds={"maxlags":3})
break_terms = ["post"]+[c+"_x_post" for c in FAC]
wald = rf.wald_test(", ".join(t+" = 0" for t in break_terms), use_f=True)
Fstat = float(np.ravel(wald.statistic)[0]); pval = float(wald.pvalue)
print("\n=== (1b) Chow test at 2020 - FULL break (all 7 coefficients) ===")
print(f"  Wald F({len(break_terms)}, {int(rf.df_resid)}) = {Fstat:.2f},  p = {pval:.3f}")
print(f"  -> full structural break is {'SIGNIFICANT' if pval<0.05 else 'not significant'} at 5%")

=== (1a) Chow test at 2020 - ALPHA break ===
  pre-2020 alpha  = +0.47%/yr
  post-2020 alpha = +3.14%/yr
  change in alpha = +2.67%/yr  (t=+1.25, p=0.213)
  -> alpha break is not significant at 5%

=== (1b) Chow test at 2020 - FULL break (all 7 coefficients) ===
  Wald F(7, 118) = 1.42,  p = 0.202
  -> full structural break is not significant at 5%


/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/statsmodels/base/model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(


In [3]:
# (2) SUP-WALD (QLR): break in the alpha at an UNKNOWN date, 15% trimming
n=len(df); lo=int(np.ceil(0.15*n)); hi=int(np.floor(0.85*n))
rows=[]
for i in range(lo, hi+1):
    brk = (df.index.to_numpy() >= i).astype("float64")
    X = sm.add_constant(df[FAC].astype("float64")); X["brk"]=brk
    r = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":3})
    rows.append((df["date"].iloc[i], r.tvalues["brk"]**2))   # Wald chi2(1) for the alpha break
qlr = pd.DataFrame(rows, columns=["date","wald"])
suprow = qlr.loc[qlr["wald"].idxmax()]
# Andrews (1993) asymptotic critical values, 1 parameter, 15% trimming
ANDREWS = {"10%":7.17, "5%":8.85, "1%":12.35}
print("=== (2) sup-Wald (QLR), alpha break at unknown date (15% trim) ===")
print(f"  sup-Wald = {suprow['wald']:.2f}  at  {suprow['date'].date()}")
print(f"  Andrews(1993) critical values (1 param, 15% trim): 10%={ANDREWS['10%']}, 5%={ANDREWS['5%']}, 1%={ANDREWS['1%']}")
verdict = "REJECT no-break" if suprow['wald']>ANDREWS['5%'] else "CANNOT reject coefficient stability"
print(f"  -> {verdict} at 5%")

=== (2) sup-Wald (QLR), alpha break at unknown date (15% trim) ===
  sup-Wald = 10.61  at  2015-10-31
  Andrews(1993) critical values (1 param, 15% trim): 10%=7.17, 5%=8.85, 1%=12.35
  -> REJECT no-break at 5%


In [4]:
# Summary table
summary = pd.DataFrame([
    {"test":"Chow 2020 (alpha break)", "statistic":f"t={ra.tvalues['post']:.2f}", "p_value":round(float(ra.pvalues['post']),3),
     "detail":f"alpha change {d_alpha*12*100:+.2f}%/yr; pre {ra.params['const']*12*100:+.2f} -> post {(ra.params['const']+d_alpha)*12*100:+.2f}%/yr",
     "significant_5pct": bool(ra.pvalues['post']<0.05)},
    {"test":"Chow 2020 (full break, 7 coeff)", "statistic":f"F={Fstat:.2f}", "p_value":round(pval,3),
     "detail":"joint break in const + 6 factor loadings", "significant_5pct": bool(pval<0.05)},
    {"test":"sup-Wald (alpha, unknown date)", "statistic":f"W={suprow['wald']:.2f}", "p_value":"vs Andrews CV 8.85 (5%)",
     "detail":f"max at {suprow['date'].date()}", "significant_5pct": bool(suprow['wald']>8.85)},
])
print(summary.to_string(index=False))
summary.to_csv(OUTPUT/"table_structural_break.csv", index=False)
print(f"\nSaved to {OUTPUT/'table_structural_break.csv'}")

                           test statistic                 p_value                                              detail  significant_5pct
        Chow 2020 (alpha break)    t=1.25                   0.213 alpha change +2.67%/yr; pre +0.47 -> post +3.14%/yr             False
Chow 2020 (full break, 7 coeff)    F=1.42                   0.202            joint break in const + 6 factor loadings             False
 sup-Wald (alpha, unknown date)   W=10.61 vs Andrews CV 8.85 (5%)                                   max at 2015-10-31              True

Saved to /Users/<wrds-username>/thesis/output/table_structural_break.csv
